# Cleanup and Installation
Run these commands to clean up previous data and install required CLI tools for the RGB sandbox demo.

# Cleanup and Installation
Run these commands to clean up previous data and install required CLI tools for the RGB sandbox demo.

In [ ]:
# Remove installed crates
rm -r bp-wallet rgb-cmd

# Install bp-wallet CLI
cargo install bp-wallet --version 0.11.1-alpha.2 --root ./bp-wallet --features=cli,hot

# Install rgb-cmd CLI
cargo install rgb-cmd --version 0.11.1-rc.5 --root ./rgb-cmd


In [ ]:
# Remove data directories and generated files
rm -fr data{0,1} wallets consignment.yaml contracts/usdt.yaml


In [ ]:
docker-compose down -v && docker-compose up -d

just setup-bitcoind


In [ ]:


echo '{"jsonrpc": "2.0", "method": "blockchain.block.header", "params": [100], "id": 0}' | netcat -w1 localhost 50001


# RGB Asset Minting and Transfer (Manual Demo)
This notebook contains the step-by-step commands for minting and transferring an RGB asset, following the manual demo instructions. The Bitcoin infrastructure is assumed to be already initialized and running.


## 1. Set up aliases and environment variables
These commands make it easier to use the RGB and wallet CLIs.


In [ ]:
# Aliases for easier CLI usage
alias bp="bp-wallet/bin/bp"
alias bphot="bp-wallet/bin/bp-hot"
alias rgb0="rgb-cmd/bin/rgb -n regtest --electrum=localhost:50001 -d data0 -w issuer"
alias rgb1="rgb-cmd/bin/rgb -n regtest --electrum=localhost:50001 -d data1 -w rcpt1"
alias bcli='docker compose exec -T bitcoind bitcoin-cli -regtest -rpcuser=polaruser -rpcpassword=polarpass'

# Environment variables
CLOSING_METHOD="opret1st"
CONSIGNMENT="consignment.rgb"
PSBT="tx.psbt"
SCHEMATA_DIR="rgb-schemas/schemata"
WALLET_PATH="wallets"
KEYCHAIN="<0;1;9>"


## 2. Prepare Bitcoin wallets
Assumes Bitcoin Core wallet is already created and funded. Remove old wallets and create new ones for issuer and receiver.

In [ ]:
# Remove old wallets if needed
rm -fr $WALLET_PATH

# Create wallet directory
mkdir -p $WALLET_PATH

# Seed password definition
export SEED_PASSWORD="seed test password"

# Issuer wallet: generate seed and derive account
bphot seed "$WALLET_PATH/0.seed" > issuer_seed_out.txt
bphot derive -N -s bip86 "$WALLET_PATH/0.seed" "$WALLET_PATH/0.derive" > issuer_derive_out.txt
account_0=$(grep -oE '\[[0-9a-f]{8}/86h/1h/0h\][^ ]+' issuer_derive_out.txt | head -n1)
descriptor_0="$account_0/$KEYCHAIN/*"

# Receiver wallet: generate seed and derive account
bphot seed "$WALLET_PATH/1.seed" > receiver_seed_out.txt
bphot derive -N -s bip86 "$WALLET_PATH/1.seed" "$WALLET_PATH/1.derive" > receiver_derive_out.txt
account_1=$(grep -oE '\[[0-9a-f]{8}/86h/1h/0h\][^ ]+' receiver_derive_out.txt | head -n1)
descriptor_1="$account_1/$KEYCHAIN/*"

echo "account_0=\"$account_0\""
echo "descriptor_0=\"$descriptor_0\""
echo "account_1=\"$account_1\""
echo "descriptor_1=\"$descriptor_1\""


## 3. Set up RGB wallets
Create RGB wallets for issuer and receiver, and import the NIA schema.

In [ ]:

# Issuer RGB wallet
rgb0 create --wpkh $descriptor_0 issuer

# Receiver RGB wallet
rgb1 create --wpkh $descriptor_1 rcpt1

# Import NIA schema into both wallets
rgb0 import $SCHEMATA_DIR/NonInflatableAsset.rgb
rgb1 import $SCHEMATA_DIR/NonInflatableAsset.rgb

# Get schema ID
schema_id=$(rgb0 schemata | grep NonInflatableAsset | awk '{print $2}')


## 4. Prepare UTXOs
Generate addresses, fund wallets, and gather outpoints for asset issuance and receiving.

In [ ]:
# Generate addresses for asset issuance and receiving
addr_issue=$(rgb0 address -k 9 | grep '&9/0' | awk '{print $2}')
if [ -z "$addr_issue" ]; then
  addr_issue=$(rgb0 address -k 9 | grep -Eo 'bcrt1[qpz0-9a-z]+' | head -n1)
fi
addr_receive=$(rgb1 address -k 9 | grep '&9/0' | awk '{print $2}')
if [ -z "$addr_receive" ]; then
  addr_receive=$(rgb1 address -k 9 | grep -Eo 'bcrt1[qpz0-9a-z]+' | head -n1)
fi
echo "addr_issue=$addr_issue"
echo "addr_receive=$addr_receive"
# Fund wallets (assumes bcli and wallet are ready)
bcli -rpcwallet=default sendtoaddress "$addr_issue" 1
bcli -rpcwallet=default sendtoaddress "$addr_receive" 1
bcli -rpcwallet=default -generate 1



In [ ]:
# Sync wallets and gather outpoints
rgb0 utxos --sync > issuer_utxos.txt
outpoint_issue=$(grep -Eo '[0-9a-f]{64}:[0-9]+' issuer_utxos.txt | head -n1)
rgb1 utxos --sync > rcpt_utxos.txt
outpoint_receive=$(grep -Eo '[0-9a-f]{64}:[0-9]+' rcpt_utxos.txt | head -n1)
echo "outpoint_issue=$outpoint_issue"
echo "outpoint_receive=$outpoint_receive"


## 5. Asset issuance
Prepare the contract file and issue the asset using the RGB CLI.

In [ ]:
# Prepare contract file for asset issuance
sed \
  -e "s/schema_id/$schema_id/" \
  -e "s/issued_supply/1000/" \
  -e "s/txid:vout/$outpoint_issue/" \
  contracts/usdt.yaml.template > contracts/usdt.yaml

# Issue the asset
rgb0 issue "ssi:issuer" contracts/usdt.yaml

# Get contract ID
contract_id=$(rgb0 contracts | grep NonInflatableAsset | awk '{print $1}')

echo $contract_id


Now we set up the contract id on the following field

In [ ]:
contract_id=rgb:GbysWHAd-xGvP2if-DSxGlFU-e~HSCo3-cxuIJhc-71vQ6ck


## 6. Transfer: Receiver generates invoice
The receiver generates an invoice to receive assets.

In [ ]:
# Receiver generates invoice for 100 units
invoice=$(rgb1 invoice --amount 100 "$contract_id")
echo $invoice


## 7. Transfer: Sender initiates asset transfer
Sender creates the consignment and PSBT for the transfer.

In [ ]:
# Sender creates consignment and PSBT
rgb0 transfer "$invoice" "data0/$CONSIGNMENT" "data0/$PSBT"

# (Optional) Inspect consignment
rgb0 inspect "data0/$CONSIGNMENT" > consignment.yaml

# Exchange consignment file to receiver
cp data0/$CONSIGNMENT data1/$CONSIGNMENT


## 8. Receiver: Validate transfer
Receiver validates the consignment before accepting.

In [ ]:
# Receiver validates consignment
rgb1 validate "data1/$CONSIGNMENT"


## 9. Sender: Sign and broadcast transaction
Sender signs, finalizes, and broadcasts the transaction after receiver validation.

In [ ]:
# Sender signs the PSBT
bphot sign -N "data0/$PSBT" "$WALLET_PATH/0.derive"

# Finalize and broadcast transaction
rgb0 finalize -p data0/$PSBT data0/${PSBT%psbt}tx


## 10. Confirm transaction and sync wallets
Confirm the transaction and update wallet states.

In [ ]:
# Confirm transaction (mine a block)
bcli -rpcwallet=default -generate 1

# Sync wallets
rgb0 utxos --sync
rgb1 utxos --sync


## 11. Receiver: Accept transfer
Receiver accepts the transfer and updates contract state.

In [ ]:
# Receiver accepts the transfer
rgb1 accept "data1/$CONSIGNMENT"

# Show updated contract state (receiver)
rgb1 state "$contract_id"

# Show updated contract state (issuer)
rgb0 state "$contract_id"


# RGB Lightning Network Demo

Now we'll work with RGB Lightning Network nodes to create channels and make payments using the RGB asset. The topology is: Alice → Bob (routing node) → Carol

## 12. Step 1: Initialize and Unlock Lightning Nodes

First, we need to initialize and unlock the three RGB Lightning nodes (Alice, Bob, and Carol).

In [ ]:
# Initialize Alice node
curl -X POST http://localhost:3001/init \
  -H "Content-Type: application/json" \
  -d '{
    "password": "alice12345678",
    "mnemonic_words": 24
  }'

# Initialize Bob node
curl -X POST http://localhost:3002/init \
  -H "Content-Type: application/json" \
  -d '{
    "password": "bob12345678",
    "mnemonic_words": 24
  }'

# Initialize Carol node
curl -X POST http://localhost:3003/init \
  -H "Content-Type: application/json" \
  -d '{
    "password": "carol12345678",
    "mnemonic_words": 24
  }'


In [ ]:
# Unlock Alice node
curl -X POST http://localhost:3001/unlock \
  -H "Content-Type: application/json" \
  -d '{
    "password": "alice12345678",
    "bitcoind_rpc_username": "polaruser",
    "bitcoind_rpc_password": "polarpass",
    "bitcoind_rpc_host": "bitcoind",
    "bitcoind_rpc_port": 18443,
    "indexer_url": "electrs:50001",
    "proxy_endpoint": "rpc://rgb-proxy:3000/json-rpc",
    "announce_addresses": []
  }'

# Unlock Bob node
curl -X POST http://localhost:3002/unlock \
  -H "Content-Type: application/json" \
  -d '{
    "password": "bob12345678",
    "bitcoind_rpc_username": "polaruser",
    "bitcoind_rpc_password": "polarpass",
    "bitcoind_rpc_host": "bitcoind",
    "bitcoind_rpc_port": 18443,
    "indexer_url": "electrs:50001",
    "proxy_endpoint": "rpc://rgb-proxy:3000/json-rpc",
    "announce_addresses": []
  }'

# Unlock Carol node
curl -X POST http://localhost:3003/unlock \
  -H "Content-Type: application/json" \
  -d '{
    "password": "carol12345678",
    "bitcoind_rpc_username": "polaruser",
    "bitcoind_rpc_password": "polarpass",
    "bitcoind_rpc_host": "bitcoind",
    "bitcoind_rpc_port": 18443,
    "indexer_url": "electrs:50001",
    "proxy_endpoint": "rpc://rgb-proxy:3000/json-rpc",
    "announce_addresses": []
  }'


## 13. Fund Lightning Nodes with Bitcoin

Get addresses from each node and fund them with Bitcoin for channel operations.

In [ ]:
# Get Alice's address
alice_addr=$(curl -s -X POST http://localhost:3001/address | jq -r '.address')
echo "Alice address: $alice_addr"

# Get Bob's address
bob_addr=$(curl -s -X POST http://localhost:3002/address | jq -r '.address')
echo "Bob address: $bob_addr"

# Get Carol's address
carol_addr=$(curl -s -X POST http://localhost:3003/address | jq -r '.address')
echo "Carol address: $carol_addr"

bcli -rpcwallet=default -generate 6

# Fund each node with 1 BTC
bcli -rpcwallet=default sendtoaddress "$alice_addr" 1
bcli -rpcwallet=default sendtoaddress "$bob_addr" 1
bcli -rpcwallet=default sendtoaddress "$carol_addr" 1

# Mine blocks to confirm
bcli -rpcwallet=default -generate 6


## 15. Create RGB Asset Channels

Create Lightning channels with RGB asset capacity. Alice will open a 500 USDT channel to Bob, and Bob will open a 300 USDT channel to Carol.

## 14. Issue RGB Asset from Alice's Lightning Node

Instead of transferring from rgb0, we'll have Alice issue a new NIA (Non-Inflatable Asset) directly from her Lightning node using the `/issueassetnia` API.

In [ ]:
# Wait for Alice's node to sync the funding transaction
echo "=== Syncing Alice's node... ==="
curl -s -X POST http://localhost:3001/sync \
  -H "Content-Type: application/json" \
  -d '{}' | jq '.'

sleep 5

# Verify Alice has Bitcoin balance before issuing asset
echo "=== Alice's Bitcoin balance ==="
curl -s -X POST http://localhost:3001/btcbalance \
  -H "Content-Type: application/json" \
  -d '{"skip_sync": false}' | jq '.'

# Create UTXOs for RGB operations
# RGB operations need specific UTXO structures, so we split the funds into multiple UTXOs
echo "=== Creating UTXOs for RGB operations ==="
curl -s -X POST http://localhost:3001/createutxos \
  -H "Content-Type: application/json" \
  -d '{
    "up_to": false,
    "num": 5,
    "size": 5000,
    "fee_rate": 2,
    "skip_sync": false
  }' | jq '.'

# Mine blocks to confirm UTXO creation
bcli -rpcwallet=default -generate 1

# Wait for confirmation and sync
sleep 3
curl -s -X POST http://localhost:3001/sync \
  -H "Content-Type: application/json" \
  -d '{}' | jq '.'

# Alice issues a new RGB NIA asset (Non-Inflatable Asset)
# This creates a new USDT-like token with 1000 units supply
echo "=== Issuing RGB asset from Alice's Lightning node ==="
alice_asset_response=$(curl -s -X POST http://localhost:3001/issueassetnia \
  -H "Content-Type: application/json" \
  -d '{
    "ticker": "USDT",
    "name": "Lightning Tether",
    "amounts": [1000],
    "precision": 0
  }')

echo "$alice_asset_response" | jq '.'

# Extract the asset_id from the response
alice_contract_id=$(echo "$alice_asset_response" | jq -r '.asset.asset_id')
echo "Alice's newly issued asset ID: $alice_contract_id"

# Override the contract_id variable to use Alice's asset for all subsequent operations
contract_id="$alice_contract_id"

# Verify the asset was issued successfully
echo "=== Alice's RGB assets ==="
curl -s -X POST http://localhost:3001/listassets \
  -H "Content-Type: application/json" \
  -d '{"filter_asset_schemas": ["Nia"]}' | jq '.'

# Check Alice's balance - should show 1000 USDT
echo "=== Alice's asset balance ==="
curl -s -X POST http://localhost:3001/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "=== Asset issuance complete - Alice has 1000 USDT in her Lightning wallet ==="


In [ ]:
# Get Bob's node info (peer pubkey and address)
bob_info=$(curl -s -X GET http://localhost:3002/nodeinfo)
bob_pubkey=$(echo "$bob_info" | jq -r '.pubkey')
bob_peer_addr="${bob_pubkey}@rln-bob:9736"
echo "Bob peer address: $bob_peer_addr"

# Get Carol's node info
carol_info=$(curl -s -X GET http://localhost:3003/nodeinfo)
carol_pubkey=$(echo "$carol_info" | jq -r '.pubkey')
carol_peer_addr="${carol_pubkey}@rln-carol:9737"
echo "Carol peer address: $carol_peer_addr"


## 14.1 Multi-Hop Lightning Demo: Alice → Bob → Carol

For this demo, we'll create a two-hop payment path:
- Alice opens a 500 USDT RGB channel to Bob
- Bob opens a 300 USDT RGB channel to Carol
- Carol creates an invoice for 100 USDT
- Alice pays Carol's invoice, routed through Bob
- Finally, we'll close all channels to see the on-chain settlement

In [ ]:
# Alice connects to Bob
echo "=== Connecting Alice to Bob ==="
curl -X POST http://localhost:3001/connectpeer \
  -H "Content-Type: application/json" \
  -d "{\"peer_pubkey_and_addr\": \"$bob_peer_addr\"}"

# Bob connects to Carol
echo "=== Connecting Bob to Carol ==="
curl -X POST http://localhost:3002/connectpeer \
  -H "Content-Type: application/json" \
  -d "{\"peer_pubkey_and_addr\": \"$carol_peer_addr\"}"

# Wait a moment for connections to establish
sleep 3

echo "=== Peer connections established ==="


In [ ]:
# Step 1: Transfer some assets to Bob on-chain so he can open a channel to Carol
echo "=== Bob needs RGB assets to open channel to Carol ==="

# Bob needs to create UTXOs first before generating RGB invoice
echo "=== Creating UTXOs for Bob to receive RGB assets ==="
curl -s -X POST http://localhost:3002/createutxos \
  -H "Content-Type: application/json" \
  -d '{
    "up_to": false,
    "num": 5,
    "size": 10000,
    "fee_rate": 2,
    "skip_sync": false
  }' | jq '.'

bcli -rpcwallet=default -generate 1
sleep 3
curl -s -X POST http://localhost:3002/sync \
  -H "Content-Type: application/json" \
  -d '{}' | jq '.'

# Now Bob creates an on-chain RGB invoice
echo "Creating RGB invoice for Bob to receive 400 USDT from Alice..."
bob_onchain_invoice_response=$(curl -s -X POST http://localhost:3002/rgbinvoice \
  -H "Content-Type: application/json" \
  -d '{
    "assignment": {
      "type": "Fungible",
      "value": 400
    },
    "min_confirmations": 1,
    "duration_seconds": 900,
    "witness": false
  }')

echo "Bob's invoice response:"
echo "$bob_onchain_invoice_response" | jq '.'

bob_recipient_id=$(echo "$bob_onchain_invoice_response" | jq -r '.recipient_id')
echo "Bob's recipient ID: $bob_recipient_id"

# Alice sends 400 USDT to Bob on-chain
echo "=== Alice sending 400 USDT to Bob on-chain ==="
curl -s -X POST http://localhost:3001/sendasset \
  -H "Content-Type: application/json" \
  -d "{
    \"asset_id\": \"$contract_id\",
    \"assignment\": {
      \"type\": \"Fungible\",
      \"value\": 400
    },
    \"recipient_id\": \"$bob_recipient_id\",
    \"donation\": true,
    \"fee_rate\": 2,
    \"min_confirmations\": 1,
    \"transport_endpoints\": [\"rpc://rgb-proxy:3000/json-rpc\"],
    \"skip_sync\": false
  }" | jq '.'

# Mine and sync
bcli -rpcwallet=default -generate 1
sleep 3

echo "=== Refreshing transfers for Bob ==="
curl -s -X POST http://localhost:3002/refreshtransfers \
  -H "Content-Type: application/json" \
  -d '{"skip_sync": false}' | jq '.'
sleep 2
curl -s -X POST http://localhost:3002/refreshtransfers \
  -H "Content-Type: application/json" \
  -d '{"skip_sync": false}' | jq '.'

echo "=== Refreshing transfers for Alice ==="
curl -s -X POST http://localhost:3001/refreshtransfers \
  -H "Content-Type: application/json" \
  -d '{"skip_sync": false}' | jq '.'

# Check balances after transfer
echo "=== Alice's balance after transfer ==="
curl -s -X POST http://localhost:3001/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "=== Bob's balance after transfer ==="
curl -s -X POST http://localhost:3002/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

# Step 2: Refresh Alice's transfers and check available balance
echo "=== Refreshing Alice's transfers after sending to Bob ==="
curl -s -X POST http://localhost:3001/refreshtransfers \
  -H "Content-Type: application/json" \
  -d '{"skip_sync": false}' | jq '.'

echo "=== Checking Alice's available balance for channel opening ==="
alice_balance=$(curl -s -X POST http://localhost:3001/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}")
echo "$alice_balance" | jq '.'

alice_spendable=$(echo "$alice_balance" | jq -r '.spendable')
echo "Alice's spendable balance: $alice_spendable USDT"

# Step 3: Alice creates UTXOs for opening channel to Bob
echo "=== Creating UTXOs for Alice's channel to Bob ==="
curl -s -X POST http://localhost:3001/createutxos \
  -H "Content-Type: application/json" \
  -d '{
    "up_to": false,
    "num": 10,
    "size": 25000,
    "fee_rate": 2,
    "skip_sync": false
  }' | jq '.'

bcli -rpcwallet=default -generate 1
sleep 3
curl -s -X POST http://localhost:3001/sync \
  -H "Content-Type: application/json" \
  -d '{}' | jq '.'

# Step 4: Alice opens RGB channel to Bob with her remaining USDT
# Alice should have 600 USDT left (1000 - 400 sent to Bob)
# We'll check her actual spendable balance and use that
if [ "$alice_spendable" -lt "500" ]; then
  echo "⚠️  Alice has less than 500 USDT spendable ($alice_spendable USDT)"
  echo "⚠️  This may be due to existing channels. Adjusting channel amount..."
  alice_channel_amount=$alice_spendable
else
  alice_channel_amount=500
fi

echo "=== Opening RGB channel from Alice to Bob ($alice_channel_amount USDT) ==="
curl -s -X POST http://localhost:3001/openchannel \
  -H "Content-Type: application/json" \
  -d "{
    \"peer_pubkey_and_opt_addr\": \"$bob_peer_addr\",
    \"capacity_sat\": 200000,
    \"push_msat\": 0,
    \"asset_amount\": $alice_channel_amount,
    \"asset_id\": \"$contract_id\",
    \"public\": true,
    \"with_anchors\": true
  }" | jq '.'

bcli -rpcwallet=default -generate 6
sleep 3

echo "=== Alice → Bob channel opened with $alice_channel_amount USDT ==="

# Step 4: Bob creates UTXOs for opening channel to Carol
# Step 6: Bob opens RGB channel to Carol with 300 USDT
curl -s -X POST http://localhost:3002/createutxos \
  -H "Content-Type: application/json" \
  -d '{
    "up_to": false,
    "num": 10,
    "size": 25000,
    "fee_rate": 2,
    "skip_sync": false
  }' | jq '.'

bcli -rpcwallet=default -generate 1
sleep 3
curl -s -X POST http://localhost:3002/sync \
  -H "Content-Type: application/json" \
  -d '{}' | jq '.'

# Step 5: Bob opens RGB channel to Carol with 300 USDT
echo "=== Opening RGB channel from Bob to Carol (300 USDT) ==="
curl -s -X POST http://localhost:3002/openchannel \
  -H "Content-Type: application/json" \
  -d "{
    \"peer_pubkey_and_opt_addr\": \"$carol_peer_addr\",
    \"capacity_sat\": 200000,
    \"push_msat\": 0,
    \"asset_amount\": 300,
    \"asset_id\": \"$contract_id\",
    \"public\": true,
    \"with_anchors\": true
  }" | jq '.'

bcli -rpcwallet=default -generate 6
sleep 3

echo "=== Bob → Carol channel opened with 300 USDT ==="
echo "=== Channel topology complete: Alice (500) → Bob (300) → Carol ==="


## 16. Verify Channel States

Check that channels are open and have the correct RGB asset balances.

In [ ]:
bcli -rpcwallet=default -generate 6
# List Alice's channels
echo "=== Alice's Channels ==="
curl -s -X GET http://localhost:3001/listchannels | jq '.'

# List Bob's channels (should show 2 channels: with Alice and with Carol)
echo "=== Bob's Channels ==="
curl -s -X GET http://localhost:3002/listchannels | jq '.'

# List Carol's channels
echo "=== Carol's Channels ==="
curl -s -X GET http://localhost:3003/listchannels | jq '.'

# Check asset balances
echo "=== Alice's USDT Balance ==="
curl -s -X POST http://localhost:3001/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "=== Bob's USDT Balance ==="
curl -s -X POST http://localhost:3002/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "=== Carol's USDT Balance ==="
curl -s -X POST http://localhost:3003/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'


## 17. Make Multi-Hop RGB Lightning Payment (Alice → Bob → Carol)

Carol will create an invoice for 100 USDT, and Alice will pay it.
The payment will be automatically routed through Bob.

In [ ]:
# Carol generates a Lightning invoice for 100 USDT
# For RGB Lightning payments, use /lninvoice (not /rgbinvoice which is for on-chain)
# The payment will route: Alice → Bob → Carol
echo "=== Carol generating Lightning RGB invoice for 100 USDT ==="
carol_invoice_response=$(curl -s -X POST http://localhost:3003/lninvoice \
  -H "Content-Type: application/json" \
  -d "{
    \"amt_msat\": 3000000,
    \"asset_id\": \"$contract_id\",
    \"asset_amount\": 100,
    \"expiry_sec\": 900
  }")

echo "Full response:"
echo "$carol_invoice_response" | jq '.'

# Extract the invoice
carol_invoice=$(echo "$carol_invoice_response" | jq -r '.invoice')
echo "Carol's Lightning invoice for 100 USDT: $carol_invoice"


In [ ]:
# Alice pays Carol's invoice (will route through Bob)
echo "=== Alice paying Carol's invoice (routing through Bob) ==="
payment_result=$(curl -s -X POST http://localhost:3001/sendpayment \
  -H "Content-Type: application/json" \
  -d "{\"invoice\": \"$carol_invoice\"}")

echo "Payment result:"
echo "$payment_result" | jq '.'

# Extract payment hash for tracking
payment_hash=$(echo "$payment_result" | jq -r '.payment_hash')
echo "Payment hash: $payment_hash"

echo "=== Payment sent! Alice → Bob → Carol (100 USDT) ==="


## 18. Verify Payment and Channel Balances

Check that the multi-hop payment succeeded. 
Expected results:
- Alice: 400 USDT in channel (started with 500, paid 100)
- Bob: 100 USDT in Alice channel, 200 USDT in Carol channel (routed 100 from Alice to Carol)
- Carol: 100 USDT in channel (received 100)

In [ ]:
# Sync all nodes before checking balances
echo "=== Syncing all nodes... ==="
curl -s -X POST http://localhost:3001/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3002/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3003/sync -H "Content-Type: application/json" -d '{}' > /dev/null
sleep 2

# Check payment status
echo "=== Payment Status ==="
curl -s -X POST http://localhost:3001/getpayment \
  -H "Content-Type: application/json" \
  -d "{\"payment_hash\": \"$payment_hash\"}" | jq '.'

# List all payments from Alice
echo "=== Alice's Payment History ==="
curl -s -X GET http://localhost:3001/listpayments | jq '.'

# Check final asset balances
echo "=== Alice's Final USDT Balance ==="
curl -s -X POST http://localhost:3001/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "=== Bob's Final USDT Balance ==="
curl -s -X POST http://localhost:3002/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "=== Carol's Final USDT Balance (should have received 100 USDT) ==="
curl -s -X POST http://localhost:3003/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

# Show updated channel states
echo "=== Channel States After Payment ==="
echo "Alice's channels:"
curl -s -X GET http://localhost:3001/listchannels | jq '.'
echo "Bob's channels:"
curl -s -X GET http://localhost:3002/listchannels | jq '.'
echo "Carol's channels:"
curl -s -X GET http://localhost:3003/listchannels | jq '.'


## 19. Close All Channels

Now we'll close all the Lightning channels to settle the RGB assets back on-chain.
This will show the final distribution of assets after the Lightning payments.

In [ ]:
echo "=== Closing All Channels ==="

# Sync all nodes first
curl -s -X POST http://localhost:3001/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3002/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3003/sync -H "Content-Type: application/json" -d '{}' > /dev/null
sleep 2

# Get channel information
alice_channels=$(curl -s -X GET http://localhost:3001/listchannels)
bob_channels=$(curl -s -X GET http://localhost:3002/listchannels)

echo "Alice's channels:"
echo "$alice_channels" | jq '.'
echo "Bob's channels:"
echo "$bob_channels" | jq '.'

# Get node pubkeys
alice_pubkey=$(curl -s http://localhost:3001/nodeinfo | jq -r .pubkey)
bob_pubkey=$(curl -s http://localhost:3002/nodeinfo | jq -r .pubkey)
carol_pubkey=$(curl -s http://localhost:3003/nodeinfo | jq -r .pubkey)

# Extract channel IDs
alice_bob_channel_id=$(echo "$alice_channels" | jq -r '.channels[0].channel_id')
bob_carol_channel_id=$(echo "$bob_channels" | jq -r ".channels[] | select(.peer_pubkey == \"$carol_pubkey\") | .channel_id")

echo "Alice→Bob channel ID: $alice_bob_channel_id"
echo "Bob→Carol channel ID: $bob_carol_channel_id"

# Close Alice→Bob channel (cooperative close)
echo ""
echo "Closing Alice→Bob channel..."
curl -s -X POST http://localhost:3001/closechannel \
  -H "Content-Type: application/json" \
  -d "{\"channel_id\": \"$alice_bob_channel_id\", \"peer_pubkey\": \"$bob_pubkey\", \"force\": false}" | jq '.'

# Close Bob→Carol channel (cooperative close)
echo ""
echo "Closing Bob→Carol channel..."
curl -s -X POST http://localhost:3002/closechannel \
  -H "Content-Type: application/json" \
  -d "{\"channel_id\": \"$bob_carol_channel_id\", \"peer_pubkey\": \"$carol_pubkey\", \"force\": false}" | jq '.'

# Mine blocks to confirm closing transactions
echo ""
echo "Mining blocks to confirm channel closures..."
bcli -rpcwallet=default -generate 10

# Sync all nodes
echo ""
echo "Syncing all nodes..."
curl -s -X POST http://localhost:3001/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3002/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3003/sync -H "Content-Type: application/json" -d '{}' > /dev/null

# Wait a bit for sync to complete
sleep 3

# Refresh RGB transfers for all nodes (twice each to ensure proper state update)
echo "Refreshing RGB transfers..."
for i in 1 2; do
  curl -s -X POST http://localhost:3001/refreshtransfers -H "Content-Type: application/json" -d '{"skip_sync": false}' > /dev/null
  curl -s -X POST http://localhost:3002/refreshtransfers -H "Content-Type: application/json" -d '{"skip_sync": false}' > /dev/null
  curl -s -X POST http://localhost:3003/refreshtransfers -H "Content-Type: application/json" -d '{"skip_sync": false}' > /dev/null
  sleep 2
done

echo "All channels closed successfully!"


## 20. Final On-Chain Asset Distribution

After closing all channels, check the final on-chain RGB asset balances for all nodes.

In [ ]:
# Check final on-chain RGB asset balances
echo "=== Final On-Chain RGB Asset Balances ==="

# Sync all nodes one more time before final balance check
echo "=== Final sync before balance verification... ==="
curl -s -X POST http://localhost:3001/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3002/sync -H "Content-Type: application/json" -d '{}' > /dev/null
curl -s -X POST http://localhost:3003/sync -H "Content-Type: application/json" -d '{}' > /dev/null
sleep 2

echo "Alice's final USDT balance:"
curl -s -X POST http://localhost:3001/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "Bob's final USDT balance:"
curl -s -X POST http://localhost:3002/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

echo "Carol's final USDT balance:"
curl -s -X POST http://localhost:3003/assetbalance \
  -H "Content-Type: application/json" \
  -d "{\"asset_id\": \"$contract_id\"}" | jq '.'

# List all RGB assets on each node
echo "=== Alice's RGB Assets ==="
curl -s -X POST http://localhost:3001/listassets \
  -H "Content-Type: application/json" \
  -d '{"filter_asset_schemas": ["Nia"]}' | jq '.'

echo "=== Bob's RGB Assets ==="
curl -s -X POST http://localhost:3002/listassets \
  -H "Content-Type: application/json" \
  -d '{"filter_asset_schemas": ["Nia"]}' | jq '.'

echo "=== Carol's RGB Assets ==="
curl -s -X POST http://localhost:3003/listassets \
  -H "Content-Type: application/json" \
  -d '{"filter_asset_schemas": ["Nia"]}' | jq '.'
